# Producion Agricola (Analisis Exploratorio de Datos)

Una vez realizados los ajustes pertinentes al conjunto de datos principal. Procedemos a realizar el analisis exploratorio de datos.
Donde conoceremos los datos y los iremos preparando para los futuros modelos planteados.

**Observación** Una de las cosas distintivas con respecto a las etapas pasadas es que si nuestro problema lo requiere conectaremos nuestros datos principales con otras fuentes. Como veremos más adelante

In [1]:
from scripts.config import PRELOADS, DATA_DIR
import os
import pickle
import pandas as pd

# Notemos que ahora estamos obteniendo la información ya normalizada.
with open(PRELOADS['AGROPROD_2'], "rb") as preprocess: 
    dgsiap = pickle.load(preprocess)

dgsiap

Comenzamos con la carga de los conjuntos de datos de Producción Agricola de parte del DGSIAP.

In [2]:
dataset = [file.open(pars=[1, {"low_memory":False, "index_col":0}]) for file in dgsiap.branch]

y la creación de la base de datos conjutnta de la producción agrícola.

In [5]:
prodagro = pd.DataFrame()

for dset in dataset:
    prodagro = pd.concat([prodagro, dset])

print(prodagro.columns)
prodagro.head(5)

Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo',
       'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida',
       'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada',
       'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio',
       'Valorproduccion'],
      dtype='str')


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


Para nuestro analisis de datos nuestro foco estará en los distintos cultivos. Entonces primero veremos los esquemas de cultivos. Para ello retiraremos las columnas denominadas como indices. 

In [4]:
prodagro.drop(columns=[c for c in prodagro.columns if c.startswith("Id")], inplace=True)
prodagro.head(5)

,Anio,Nomestado,Nomddr,Nomcader,Nommunicipio,Nomcicloproductivo,Nommodalidad,Nomunidad,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
0,2025,Aguascalientes,Aguascalientes,Aguascalientes,Aguascalientes,Otoño-Invierno,Riego,Tonelada,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,Aguascalientes,Aguascalientes,Aguascalientes,Aguascalientes,Otoño-Invierno,Riego,Tonelada,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,Aguascalientes,Aguascalientes,Aguascalientes,Aguascalientes,Primavera-Verano,Riego,Tonelada,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,Aguascalientes,Aguascalientes,Aguascalientes,Aguascalientes,Primavera-Verano,Riego,Tonelada,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,Aguascalientes,Aguascalientes,Aguascalientes,Aguascalientes,Primavera-Verano,Riego,Tonelada,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


Viendo la producción agricola por cada ciclo productivo. Primero veremos los estados que poseen cada uno de los tipos 

In [6]:
cycle_ent = prodagro.groupby("Nomcicloproductivo")['Nomestado'].unique()
base_cycle = set(cycle_ent["Perennes"])
warm_cycle = set(cycle_ent["Primavera-Verano"])
cold_cycle = set(cycle_ent["Otoño-Invierno"])
print("Base", len(base_cycle))
print("Calido", len(warm_cycle))
print("Frio", len(cold_cycle))


Base 33
Calido 33
Frio 33


Como el ciclo productivo se encuentra en todos los estados. Seguiremos con los cultivos. Abordando 2 partes importantes: La Tasa de Éxito y el Valor de la Producción. 

* El valor de la producción, variable en el conjunto de datos, define el valor de la producción total de un cultivo dentro de un estado.
* La Tasa de éxito, calculada a partir del conjunto de datos, mide que tan efectivo es el cultivo en un estado. 

## Tasa de Éxito. 

La Tasa de Exito se calcula mediante la siguiente fórmula

$$
    \alpha = \dfrac{\text{Cos}}{\text{Sem}},
$$
donde $\text{Cos}$ son las hectáreas cosechadas y $\text{Sem}$ las hectáreas sembradas. Notemos que $\alpha$ no tiene unidades y que varia en función del cultivo y el estado.

$$
    \alpha \equiv \alpha(e,c)
$$

Primero, veremos como afecta las hectáreas sembreadas y cosechadas con respecto a la tasa de éxito en los paises principales, es decir, los paises con el mayor número de hectáreas. Entonces, comenzaremos agrupando por cultivo y estado nuestras variables de interés.

In [7]:
crop_ent = prodagro.groupby(["Nomcultivo",'Nomestado'])[["Cosechada", "Sembrada", "Siniestrada"]].sum()
crop_ent

Cosechada  Sembrada  Siniestrada
Nomcultivo        Nomestado                                            
Aceituna          Aguascalientes           20.00     55.00          0.0
                  Baja California       47770.60  91926.75        104.0
                  Baja California Sur     108.00    164.00          0.0
                  Chihuahua                 0.00     16.50          0.0
                  Ciudad de México         11.00     37.50          0.0
...                                          ...       ...          ...
Árbol de navidad  Puebla                 1140.50   1818.37          0.0
Árbol de navidad  Ciudad de México         63.30    155.60          0.0
                  Guanajuato              762.00    762.00          0.0
                  México                 2975.50   3932.50          0.0
                  Puebla                 1688.73   2218.80          0.0

[3593 rows x 3 columns]

Ahora, haremos una comparación entre los estas con más hectarias cosechadas y con más hectáreas sembradas. Primero extraremos las primeras 3 posiciones por cultivo.  

In [8]:
top_cosechada = {}
top_sembrada = {}

for crop in prodagro['Nomcultivo'].unique():
        df_cosechada = crop_ent.loc[crop, :].sort_values('Cosechada', ascending=False).head(3)
        df_sembrada = crop_ent.loc[crop, :].sort_values('Sembrada', ascending=False).head(3)
        top_cosechada[crop] = df_cosechada.index.tolist()
        top_sembrada[crop] = df_sembrada.index.tolist()


Estos, los pasaremos a tablas para medir los cambios.

In [9]:
top_cosechada = {}
top_sembrada = {}

for crop in prodagro['Nomcultivo'].unique():
        df_cosechada = crop_ent.loc[crop, :].sort_values('Cosechada', ascending=False).head(3)
        df_sembrada = crop_ent.loc[crop, :].sort_values('Sembrada', ascending=False).head(3)
        top_cosechada[crop] = df_cosechada.index.tolist()
        top_sembrada[crop] = df_sembrada.index.tolist()

top_cos = pd.DataFrame(index=[1,2,3], columns= top_cosechada.keys())
top_sem = pd.DataFrame(index=[1,2,3], columns= top_sembrada.keys())

for c in top_cos.columns:
    cos = top_cosechada[c]
    sem = top_sembrada[c]
    if len(cos)<3:
        while len(cos)<3:
            cos.append("")
    if len(sem)<3:
        while len(sem)<3:
            sem.append("")
    top_cos.loc[:, c] = top_cosechada[c]
    top_sem.loc[:, c] = top_sembrada[c]


Ahora revisaremos los cambios entre las hectáreas sembradas y cosechadas.

In [11]:
vars_top = pd.DataFrame(columns=["Pos_Sem", "Est_Sem", "Est_Cos", "Cultivo"])

k = 0
for c in top_sem:
    for n, r in enumerate(top_sem[c]):
        if r!= top_cos[c][n+1]:
            vars_top.loc[k, :] = [n+1, r, top_cos[c][n+1], c]
            k += 1
vars_top

,Pos_Sem,Est_Sem,Est_Cos,Cultivo
0,2,Sinaloa,Puebla,Calabacita
1,3,Puebla,Sinaloa,Calabacita
2,1,Guanajuato,Baja California,Cebolla
3,2,Baja California,Guanajuato,Cebolla
4,3,Aguascalientes,Durango,Maíz forrajero en verde
...,...,...,...,...
83,2,Sonora,Michoacán,Trigo forrajero achicalado
84,3,Michoacán,Sonora,Trigo forrajero achicalado
85,1,Baja California,Chihuahua,Trigo forrajero seco
86,2,Chihuahua,Sonora,Trigo forrajero seco


Primero, verificaremos la probabilidad de que a pesar de los cambios de posición los estados se mantengan en el podio.

In [12]:
var_crops = vars_top['Cultivo'].unique()
nvar_crops = len(var_crops)
print(nvar_crops)
stable_top = 0
for crop in var_crops:
    crop_t = vars_top[vars_top['Cultivo'] == crop]
    set_sem = set(crop_t['Est_Sem'].to_list())
    set_cos = set(crop_t['Est_Cos'].to_list())
    if set_cos == set_sem:
        stable_top += 1
print(f"Proporción de Tops que se mantuvieron {stable_top / nvar_crops * 100}%")

50
Proporción de Tops que se mantuvieron 52.0%


Entonces, aproximadamente la mitad de los conjuntos de estados mayoritarios se mantuvieron en el podio. Esto significa que aproximadamente 20 estados subieron de la cuarta posición a la tercera desplazando al previo tercero. Ahora realizarmos un analisis simular pero agregando la tasa de éxito. 

In [13]:
crop_ent['TasaExito'] = crop_ent['Cosechada'] / crop_ent['Sembrada']
crop_ent

Cosechada  Sembrada  Siniestrada  \
Nomcultivo        Nomestado                                               
Aceituna          Aguascalientes           20.00     55.00          0.0   
                  Baja California       47770.60  91926.75        104.0   
                  Baja California Sur     108.00    164.00          0.0   
                  Chihuahua                 0.00     16.50          0.0   
                  Ciudad de México         11.00     37.50          0.0   
...                                          ...       ...          ...   
Árbol de navidad  Puebla                 1140.50   1818.37          0.0   
Árbol de navidad  Ciudad de México         63.30    155.60          0.0   
                  Guanajuato              762.00    762.00          0.0   
                  México                 2975.50   3932.50          0.0   
                  Puebla                 1688.73   2218.80          0.0   

                                       TasaExito  
Nomcultivo        Nomestado                       
Aceituna          Aguascalientes        0.363636  
                  Baja California       0.519659  
                  Baja California Sur   0.658537  
                  Chihuahua             0.000000  
                  Ciudad de México      0.293333  
...                                          ...  
Árbol de navidad  Puebla                0.627210  
Árbol de navidad  Ciudad de México      0.406812  
                  Guanajuato            1.000000  
                  México                0.756643  
                  Puebla                0.761101  

[3593 rows x 4 columns]

In [14]:
top_exito = {}
for crop in prodagro['Nomcultivo'].unique():
        df_exito = crop_ent.loc[crop, :].sort_values('TasaExito', ascending=False).head(3)
        top_exito[crop] = df_exito.index.tolist()

In [15]:
top_ext = pd.DataFrame(index=[1,2,3], columns= top_exito.keys())

for c in top_ext.columns:
    ext = top_exito[c]
    if len(ext)<3:
        while len(ext)<3:
            ext.append("")
    top_ext.loc[:, c] = top_exito[c]

top_ext

,Avena forrajera en verde,Brócoli,Calabacita,Cebolla,Chile verde,Fresa,Maíz forrajero en verde,Maíz grano,Sorgo forrajero en verde,Tomate rojo (jitomate),...,Nectarina,Viveros,Centeno forrajero seco,Flores,Damazco,Gypsophilia,Semilla de colza,Flor cundeamor,Berro,Almácigo
1,Guerrero,Coahuila,Ciudad de México / DF,Colima,Ciudad de México,Coahuila,Ciudad de México,Ciudad de México,Chiapas,Ciudad de México / DF,...,Baja California,Chihuahua,Chihuahua,Ciudad de México / DF,Durango,Michoacán,Tlaxcala,Morelos,Morelos,Nayarit
2,Baja California Sur,Sinaloa,Morelos,Yucatán,Morelos,Puebla,Campeche,Chiapas,Morelos,Tlaxcala,...,,,,Michoacán,,,,,,
3,Nayarit,Nuevo León,Ciudad de México,Veracruz,Chiapas,Sonora,Guerrero,Tlaxcala,Guerrero,Morelos,...,,,,Nayarit,,,,,,


In [17]:
vars_top_semext = pd.DataFrame(columns=["Pos_Sem", "Est_Sem", "Est_TasaExito", "Cultivo"])

k = 0
for c in top_sem:
    for n, r in enumerate(top_sem[c]):
        if r!= top_ext[c][n+1]:
            vars_top_semext.loc[k, :] = [n+1, r, top_ext[c][n+1], c]
            k += 1
vars_top_semext.head(5)

,Pos_Sem,Est_Sem,Est_TasaExito,Cultivo
0,1,Chihuahua,Guerrero,Avena forrajera en verde
1,2,Durango,Baja California Sur,Avena forrajera en verde
2,3,Zacatecas,Nayarit,Avena forrajera en verde
3,1,Guanajuato,Coahuila,Brócoli
4,2,Puebla,Sinaloa,Brócoli


In [18]:
var_crops = vars_top_semext['Cultivo'].unique()
nvar_crops = len(var_crops)
print(nvar_crops)
stable_top = 0
for crop in var_crops:
    crop_t = vars_top_semext[vars_top_semext['Cultivo'] == crop]
    set_sem = set(crop_t['Est_Sem'].to_list())
    set_cos = set(crop_t['Est_TasaExito'].to_list())
    if set_cos == set_sem:
        stable_top += 1
print(f"Proporción de Tops que se mantuvieron {stable_top / nvar_crops * 100}%")

317
Proporción de Tops que se mantuvieron 28.391167192429023%


In [20]:
vars_top_cosext = pd.DataFrame(columns=["Pos_Sem", "Est_Cos", "Est_TasaExito", "Cultivo"])

k = 0
for c in top_cos:
    for n, r in enumerate(top_cos[c]):
        if r!= top_ext[c][n+1]:
            vars_top_cosext.loc[k, :] = [n+1, r, top_ext[c][n+1], c]
            k += 1
vars_top_cosext.head(5)

,Pos_Sem,Est_Cos,Est_TasaExito,Cultivo
0,1,Chihuahua,Guerrero,Avena forrajera en verde
1,2,Durango,Baja California Sur,Avena forrajera en verde
2,3,Zacatecas,Nayarit,Avena forrajera en verde
3,1,Guanajuato,Coahuila,Brócoli
4,2,Puebla,Sinaloa,Brócoli


In [21]:
var_crops = vars_top_cosext['Cultivo'].unique()
nvar_crops = len(var_crops)
print(nvar_crops)
stable_top = 0
for crop in var_crops:
    crop_t = vars_top_cosext[vars_top_cosext['Cultivo'] == crop]
    set_sem = set(crop_t['Est_Cos'].to_list())
    set_cos = set(crop_t['Est_TasaExito'].to_list())
    if set_cos == set_sem:
        stable_top += 1
print(f"Proporción de Tops que se mantuvieron {stable_top / nvar_crops * 100}%")

313
Proporción de Tops que se mantuvieron 28.753993610223645%


Observamos que menos del 30% con grandes cantidades de hectáreas de cultivo cosechada y sembrada mantienen alguna posición al considerar la tasa de éxito. Mostrando que existe algo más significativo que las hectáreas sembradas y cosechadas con respecto a la tasa de éxito de un cultivo en algun estado. 

Como adicional dejaremos las funciones que simplifican el proceso realizado anteriormente.

In [22]:
def variable_top(vards:list, top=3):
    list_tops = {}
    list_dfs = {}
    var_ent = prodagro.groupby(["Nomcultivo",'Nomestado'])[vards].sum()
    for var in vards:
        top_var = {}
        for crop in prodagro['Nomcultivo'].unique():
            df_crop = var_ent.loc[crop]
            df_var = df_crop.sort_values(var, ascending=False).head(top)
            top_var[crop] = df_var.index.tolist()
        list_tops[var]=top_var
    for var, vdict in list_tops.items():
        summ_var = pd.DataFrame(index=[1,2,3], columns= vdict.keys())
        for cvar in summ_var:
            v = list_tops[var][cvar]
            if len(v)<3:
                while len(v)<3:
                    v.append("")
            summ_var.loc[:, cvar] = v
        list_dfs[var] = summ_var
    return list_dfs
             

def variation_top(varpair):
    var = list(varpair.keys())
    compare_top = pd.DataFrame(columns=[
        f"Pos_{var[0][:3]}",
        f"Est_{var[0][:3]}",
        f"Est_{var[1][:3]}", "Cultivo"])
    k = 0
    for c in varpair[var[0]]:
        for n, r in enumerate(varpair[var[0]][c]):
            if r!= varpair[var[1]][c][n+1]:
                compare_top.loc[k, :] = [n+1, r, varpair[var[1]][c][n+1], c]
                k += 1
    var_crops = compare_top['Cultivo'].unique()
    nvar_crops = len(var_crops)
    stable_top = 0
    for crop in var_crops:
        crop_t = compare_top[compare_top['Cultivo'] == crop]
        set_sem = set(crop_t[f"Est_{var[0][:3]}"].to_list())
        set_cos = set(crop_t[f"Est_{var[1][:3]}"].to_list())
        if set_cos == set_sem:
            stable_top += 1
    compare_rate = stable_top / nvar_crops
    print(f"Proporción de Tops que se mantuvieron {compare_rate * 100}%")
    return compare_top, compare_rate



## Valor Producción.

De forma similar a lo visto con la tasa de éxito. Verificaremos como influyen las hectáreas sembradas y cosechadas en el valor de la producción final.

In [23]:
crop_value_ent = variable_top(["Cosechada", "Sembrada", "Precio", "Valorproduccion"])


Ahora comparemos las hectareas cosechadas y las sembradas con respecto al valor de producción.

In [24]:
cos_vlrp = {}
cos_vlrp['Cosechada'] = crop_value_ent['Cosechada']
cos_vlrp['Valorproducción'] = crop_value_ent['Valorproduccion']

table, perc = variation_top(cos_vlrp)

Proporción de Tops que se mantuvieron 49.43181818181818%


In [43]:
sem_vlrp = {}
sem_vlrp['Sembrada'] = crop_value_ent['Sembrada']
sem_vlrp['Valorproducción'] = crop_value_ent['Valorproduccion']

table, perc = variation_top(sem_vlrp)

Proporción de Tops que se mantuvieron 48.421052631578945%


In [25]:
pco_vlrp = {}
pco_vlrp['Precio'] = crop_value_ent['Precio']
pco_vlrp['Valorproducción'] = crop_value_ent['Valorproduccion']

table, perc = variation_top(pco_vlrp)

Proporción de Tops que se mantuvieron 36.04651162790697%


Vemos que los estados con mayor cantidad de hectáreas cosechadas tuvieron mayor persistencia con respecto al valor de producción. Esto tiene lógica puesto que 

$$
    V_aP=V_oP\cdot PV,
$$
donde $P_oV$ es el volumen de producción. Sabemos que $V_oP$ es directamente proporcional a las hectáreas cosechadas, $\text{Cos}$.

$$
    V_oP=k\text{Cos}\equiv \text{Cos}\propto V_oP. 
$$

Esto último nos permite predecir el valor de producción mediante 

$$
    V_aP\approx \tilde{k}\text{Cos}\tilde{PV} \approx \tilde{k}\tilde{\alpha}\text{Sem}\tilde{PV},
$$

donde $\tilde{k}$ es el estimador del multiplicador de la cosecha, cuanto cosechamos por hectarea, $\tilde{PV}$ el estimador del precio de venta y $\alpha$ es el estimador de la tasa de éxito. 